In [0]:
#sorting , union ,aggregations
# Emp Data & Schema

emp_data_1 = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"]
]

emp_data_2 = [
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"


emp_data_1=spark.createDataFrame(data=emp_data_1, schema=emp_schema)
emp_data_2=spark.createDataFrame(data=emp_data_2, schema=emp_schema)
emp_data_1.show()
emp_data_2.show()




In [0]:
#union and union all
emp=emp_data_1.union(emp_data_2)
# Example of unionByName (columns matched by name, not position)
emp_by_name = emp_data_1.unionByName(emp_data_2)
display(emp_by_name)

#sort the data emp on desc salary
from pyspark.sql.functions import desc ,col,count,sum

# Both lines sort the DataFrame 'emp' by salary in descending order.
# The first uses sort() with desc(), the second uses orderBy() with col().desc().
# Functionally, they are equivalent in Spark; sort() is an alias for orderBy().
emp = emp.sort(desc("salary"))
from pyspark.sql.functions import col

emp = emp.orderBy(col("salary").desc())
#aggrations
emp_count=emp.groupBy("department_id").agg(count("employee_id"))
# emp_count.show()

dept_salary = emp.groupBy("department_id").agg(
    sum(col("salary")).alias("total_salary"),
    count("employee_id").alias("employee_count")
).where("employee_count >2")
display(dept_salary)


In [0]:
#distinct emp
# display(emp.distinct())
emp_dept=emp.select("department_id").distinct()
emp_dept.show()

In [0]:
#max salary --select *, max(salary) over (partition by department_id order by salary desc) as max_salary from emp
from pyspark.sql.functions import col, desc, max,row_number
from pyspark.sql.window import Window

window_spec = Window.partitionBy("department_id")
emp_1 = emp.withColumn("max_salary", max(col("salary")).over(window_spec))
emp_1 = emp_1.orderBy(col("max_salary").desc())
# display(emp_1)

# Window Functions - 2nd highest salary of each department
# select *, row_number() over(partition by department_id order by salary desc) as rn from emp_unique where rn = 2

# Using WindowSpec and row_number to get 2nd highest salary per department
window_spec_=Window.partitionBy(col("department_id")).orderBy(col("salary").desc())
rn=row_number().over(window_spec_)
emp_2=emp.withColumn("rn" , rn).where("rn=2")
# emp_2.show()

# Using expr to achieve the same result for 2nd highest salary per department
from pyspark.sql.functions import expr

# expr allows us to use SQL-like syntax for window functions
emp_2_expr = emp.withColumn(
    "rn",
    expr("row_number() over (partition by department_id order by salary desc)")
).where("rn=2")
display(emp_2_expr)

emp.createOrReplaceTempView("emp")

emp_2_sql = spark.sql("""
SELECT *
FROM (
    SELECT *,
           row_number() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rn
    FROM emp
) t
WHERE rn = 2
""")
display(emp_2_sql)



In [0]:
# 'viot' could represent valid IoT data, 'unviot' invalid or anomalous data
data = [
    ("John", "2023", 100),
    ("John", "2024", 150),
    ("Mary", "2023", 200),
    ("Mary", "2024", 250)
]

df = spark.createDataFrame(data, ["employee", "year", "sales"])

pivot_df = (
    df.groupBy("employee")
      .pivot("year")
      .sum("sales")
)

display(pivot_df)
# unpivot the pivot_df to long format
unpivot_df = pivot_df.unpivot("employee", ["2023", "2024"], "year", "sales")
display(unpivot_df)
pivot_df.show()

monthly_df = sales_df.selectExpr(
    "store",
    """
    stack(
        3,
        'Jan', Jan,
        'Feb', Feb,
        'Mar', Mar
    ) as (month, sales)
    """
)

monthly_df.show()

In [0]:
# DEDUPLICATION PATTERN — keep latest row per employee
w_latest = Window.partitionBy('employee_id').orderBy(col('hire_date').desc())
emp_latest = emp.withColumn('_rn', row_number().over(w_latest)).filter(col('_rn') == 1).drop('_rn')

# Analytic: access adjacent rows for salary history
w_emp_salary = Window.partitionBy('employee_id').orderBy('hire_date')
emp = emp.withColumn('prev_salary', lag('salary', 1).over(w_emp_salary))
emp = emp.withColumn('next_hire_date', lead('hire_date', 1).over(w_emp_salary))

# Month-over-month salary growth (assuming 'month' and 'salary' columns exist)
w_mom = Window.partitionBy('department_id').orderBy('month')
emp = emp.withColumn('prev_salary', lag('salary', 1).over(w_mom))
emp = emp.withColumn('mom_pct', round((col('salary') - col('prev_salary')) / col('prev_salary') * 100, 2))

# Running total of salary per employee
w_running = Window.partitionBy('employee_id').orderBy('hire_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
emp = emp.withColumn('running_total_salary', sum('salary').over(w_running))
-
# 7-row rolling average salary per department
w_7row = Window.partitionBy('department_id').orderBy('hire_date').rowsBetween(-6, 0)
emp = emp.withColumn('rolling_7row_avg_salary', avg('salary').over(w_7row))